# 🇹🇷 Turkspell: Google Colab GPU Geliştirme ve Morfolojik Analiz Pipeline'ı

Bu notebook, **Turkspell Türkçe Hunspell Sözlüğü**'nü yerel bilgisayarınızda bir GPU'ya ihtiyaç duymadan **Google Colab'ın bulut GPU'ları (L4 veya T4)** üzerinden geliştirmek, eğitmek ve kelime dağarcığını zenginleştirmek için hazırlanmıştır.

> **💡 Google AI Pro Kullanıcıları İçin Not:**
> Google AI Pro aboneliğinizle gelen **200 Colab Compute Units (CCU)** sayesinde bu notebook'taki tüm adımları **0 TL ek ücretle** çalıştırabilirsiniz. Bu pipeline'ın tam bir döngüsü yalnızca ~2-4 CCU harcar.

### 📌 İçerik:
1. **Adım 1:** GPU Durumu ve Google Drive Bağlantısı
2. **Adım 2:** Bağımlılıkların ve Hunspell Linux Araçlarının Kurulumu
3. **Adım 3:** Turkspell Deposu ve Sözlük Dosyalarının Hazırlanması
4. **Adım 4:** Türkçe Derlemden Bilinmeyen Sözcüklerin Taranması (OOV Discovery)
5. **Adım 5:** Qwen-2.5-7B ile GPU Hızlandırmalı Morfolojik Analiz ve Kök Çıkarma
6. **Adım 6:** Dilbilgisi Kuralları İçin QLoRA İnce Ayar (Fine-Tuning - İsteğe Bağlı)
7. **Adım 7:** Sözlüğün Derlenmesi (`tr.aff`, `tr.dic`) ve Doğrulama Testleri
8. **Adım 8:** Üretilen Sözlük ve JSON Çıktılarının İndirilmesi

## ⚡ Adım 1: GPU Donanımı Kontrolü ve Google Drive Bağlantısı

Colab menüsünden **Çalışma Zamanı (Runtime) -> Çalışma zamanı türünü değiştir (Change runtime type)** seçeneğine gidin.
- Donanım hızlandırıcı: **GPU**
- GPU Türü: **L4 GPU** (Önerilen, 24 GB VRAM) veya **T4 GPU** (15 GB VRAM) seçin.

In [ ]:
# GPU Bilgisini Kontrol Et
!nvidia-smi

import torch
print(f"CUDA Kullanılabilir mi: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Modeli: {torch.cuda.get_device_name(0)}")
    print(f"Toplam VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Çıktıları güvende tutmak için Google Drive'ı bağlayın
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/turkspell_outputs'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Google Drive çıktı klasörü hazır: {DRIVE_OUTPUT_DIR}")

## 📦 Adım 2: Bağımlılıkların ve Hunspell Linux Araçlarının Kurulumu

In [ ]:
# Sistem Hunspell kütüphaneleri
!apt-get update -qq
!apt-get install -y -qq hunspell libhunspell-dev

# PyTorch, Hugging Face, BitsAndBytes ve PEFT kütüphaneleri
!pip install -q transformers peft bitsandbytes accelerate trl datasets pytest zstandard

## 📂 Adım 3: Turkspell Deposunu İndirme / Hazırlama

In [ ]:
import os

%cd /content
if not os.path.exists('/content/turkspell'):
    !git clone https://github.com/selimsum/turkspell.git
    %cd /content/turkspell
else:
    %cd /content/turkspell
    !git pull

print("Turkspell çalışma dizini:", os.getcwd())

## 🔍 Adım 4: Türkçe Derlemden Bilinmeyen Sözcüklerin Taranması

Bu adımda Türkçe Wikipedia veya OSCAR derleminden metin akışı yapılarak Turkspell'in henüz tanımadığı aday sözcükler (`hunspell -l` ile) filtrelenir.

In [ ]:
import re
import subprocess
from collections import Counter
from datasets import load_dataset

print("Hugging Face üzerinden Türkçe Wikipedia akışı başlatılıyor...")
wiki_dataset = load_dataset("wikimedia/wikipedia", "20231101.tr", split="train", streaming=True)

TURKISH_WORD_RE = re.compile(r'^[abcçdefgğhıijklmnoöprsştuüvyzâîû]+$', re.IGNORECASE)
token_counter = Counter()
sample_limit = 5000  # İhtiyaca göre 50.000'e çıkarabilirsiniz

for i, item in enumerate(wiki_dataset):
    text = item.get('text', '')
    words = re.findall(r'[a-zA-ZçÇğĞıİöÖşŞüÜâîûÂÎÛ]+', text)
    for w in words:
        if len(w) >= 3 and not any(c in 'qwxQWX' for c in w):
            w_lower = w.replace('I', 'ı').replace('İ', 'i').replace('Î', 'î').replace('Â', 'â').replace('Û', 'û').lower()
            token_counter[w_lower] += 1
    if i >= 1500:  # 1500 makale yeterli bir havuz sunar
        break

# Sık geçen adayları seç (en az 3 kez geçmiş)
candidates = [w for w, count in token_counter.most_common() if count >= 3][:sample_limit]
print(f"Toplam {len(candidates)} tekil aday toplandı.")

# Hunspell ile tanınmayanları tespit et
with open("temp_candidates.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(candidates))

p = subprocess.run(["hunspell", "-d", "tr", "-l", "temp_candidates.txt"], capture_output=True, text=True, encoding="utf-8")
undetected = [w.strip() for w in p.stdout.splitlines() if w.strip()]
print(f"Hunspell tarafından tanınmayan (OOV) geçerli Türkçe aday sayısı: {len(undetected)}")

with open("undetected_words.txt", "w", encoding="utf-8") as f:
    f.write("word\tfreq\n")
    for w in undetected:
        f.write(f"{w}\t{token_counter[w]}\n")
print("Adaylar 'undetected_words.txt' dosyasına kaydedildi.")

## 🧠 Adım 5: Qwen-2.5-7B ile GPU Hızlandırmalı Morfolojik Analiz

Bu adımda açık kaynaklı `Qwen/Qwen2.5-7B-Instruct` modeli Colab GPU'suna 4-bit (NF4) olarak yüklenir.
Model aday kelimeleri analiz ederek:
- Doğru Kök / Lemma (fiiller için mastar hali: *yapmak*, isimler için yalın kök)
- Sözcük Türü (POS: *Noun, Verb, Adjective, Adverb*)
- Ses Olayları (Ünsüz yumuşaması: `Voicing`, Ünlü düşmesi: `LastVowelDrop`, İyelikli bileşik isim: `CompoundP3sg`)
tespit eder ve anında JSON olarak kaydeder.

In [ ]:
output_file = f"{DRIVE_OUTPUT_DIR}/colab_parsed_candidates.json"

# GPU Morfolojik Ayrıştırıcıyı Çalıştır
!python training/llm_morphology_parser.py \
    --model Qwen/Qwen2.5-7B-Instruct \
    --input undetected_words.txt \
    --limit 300 \
    --batch-size 15 \
    --load-in-4bit \
    --allow-new-lemmas \
    --output {output_file}

print(f"\nİşlem tamamlandı! Sonuçlar Google Drive'ınıza kaydedildi: {output_file}")

## 🎓 Adım 6: Dilbilgisi Kural Üreteci İçin QLoRA İnce Ayar (Opsiyonel)

Eğer modelin Turkspell'in kural sentaksını (`generate_grammar_rules.py`) öğrenmesini istiyorsanız, `training/train.py` betiğini GPU üzerinde çalıştırabilirsiniz.
L4 GPU üzerinde 3 epoch eğitim ~45-60 dakika sürer ve ~2-3 CCU tüketir.

In [ ]:
# Eğitim veri kümesini oluştur
!python training/generate_training_data.py

# QLoRA SFT Eğitimini Başlat (İsteğe bağlı)
# !python training/train.py --output-dir {DRIVE_OUTPUT_DIR}/qwen_adapter --epochs 3

## 🔨 Adım 7: Sözlüğü Derleme ve Linux Benchmark Testleri

In [ ]:
# Sözlüğü derle (tr.aff ve tr.dic oluştur)
!python tools/build_v06.py

# Otomasyon testlerini çalıştır
!pytest tests/ -v -k "not slow"

## 📥 Adım 8: Çıktıları Paketleme ve İndirme

In [ ]:
import shutil
from google.colab import files

# Üretilen dosyaları bir zip paketinde topla
archive_name = "/content/turkspell_colab_results.zip"
!zip -r /content/turkspell_colab_results.zip tr.aff tr.dic undetected_words.txt

# Google Drive'a kopyala
shutil.copy("/content/turkspell_colab_results.zip", f"{DRIVE_OUTPUT_DIR}/")
print(f"Sonuç arşivi Google Drive'a yedeklendi: {DRIVE_OUTPUT_DIR}/turkspell_colab_results.zip")

# Tarayıcı üzerinden doğrudan bilgisayarınıza indirin
files.download("/content/turkspell_colab_results.zip")